In [ ]:
# import os
# import pandas as pd
# import pydicom
# from dicompylercore import dicomparser, dvhcalc
# import warnings

# # Suppress minor DICOM warnings to keep output clean
# warnings.filterwarnings("ignore")

# # --- CONFIGURATION ---
# # Path to your processed local data
# # Ensure this matches your actual folder structure:
# # Thesis_Project -> Data -> Local_Data -> Processed -> [Patient_Folders]
# data_path = '../Data/Local_Data/Processed/' 

# # Output filename
# output_csv = '../Data/local_dosiomics_batch_results.csv'

# print(f"Configuration Set.")
# print(f"Reading data from: {os.path.abspath(data_path)}")

Configuration Set.
Reading data from: d:\Thesis_Project\Data\Local_Data\Processed


In [ ]:
# def find_lung_roi_robust(structure_dict):
#     """
#     Intelligently searches for the 'Lungs' structure in a dictionary of ROIs.
#     Handles variations in naming (case, spaces, underscores).
#     """
#     # Priority list of names to look for
#     target_names_priority = [
#         'lungs_combined', 'lungs-combined', 
#         'lungs_total', 'lung_total', 'total_lung', 'total lung', 
#         'lungs', 'lung', 'lungs(total)', 'whole_lung'
#     ]
    
#     # 1. Normalize all available structure names in the file
#     available_rois = {}
#     for key, struct in structure_dict.items():
#         # clean the name: lowercase, remove special chars
#         clean_name = struct['name'].lower().strip().replace(' ', '').replace('_', '').replace('-', '')
#         available_rois[key] = clean_name

#     # 2. Check for matches against our priority list
#     for target in target_names_priority:
#         clean_target = target.replace(' ', '').replace('_', '').replace('-', '')
        
#         for key, clean_struct_name in available_rois.items():
#             if clean_target == clean_struct_name:
#                 # Found a match! Return the original ID and Name
#                 return key, structure_dict[key]['name']
    
#     return None, None

In [ ]:
# results_list = []
# skipped_patients = []

# print(f"--- Starting Batch Extraction ---")

# # Verify the data path exists
# if not os.path.exists(data_path):
#     print(f"[ERROR] Data path not found: {data_path}")
# else:
#     patient_folders = [f for f in os.listdir(data_path) if os.path.isdir(os.path.join(data_path, f))]
#     print(f"Found {len(patient_folders)} patient folders. Processing...\n")

#     for i, patient_id in enumerate(patient_folders):
#         patient_dir = os.path.join(data_path, patient_id)
        
#         # Initialize file paths
#         rtstruct_path = None
#         rtdose_path = None
        
#         # 1. Scan folder for RTSTRUCT and RTDOSE
#         for f in os.listdir(patient_dir):
#             full_path = os.path.join(patient_dir, f)
#             try:
#                 # Quick header check
#                 dcm = pydicom.dcmread(full_path, stop_before_pixels=True)
#                 if dcm.Modality == 'RTSTRUCT':
#                     rtstruct_path = full_path
#                 elif dcm.Modality == 'RTDOSE':
#                     rtdose_path = full_path
#             except:
#                 continue # Skip non-dicom files
        
#         # 2. Process if both files are found
#         if rtstruct_path and rtdose_path:
#             try:
#                 # Load Structure Set
#                 rtss = dicomparser.DicomParser(rtstruct_path)
#                 structures = rtss.GetStructures()
                
#                 # Find Lungs
#                 roi_id, roi_name = find_lung_roi_robust(structures)
                
#                 if roi_id:
#                     print(f"[{i+1}/{len(patient_folders)}] {patient_id}: Found '{roi_name}'")
                    
#                     # Calculate DVH
#                     dvh = dvhcalc.get_dvh(rtstruct_path, rtdose_path, roi_id)
                    
#                     # Extract Features
#                     metrics = {
#                         'PatientID': patient_id,
#                         'Structure_Name': roi_name,
#                         'Mean_Dose_Gy': dvh.mean,
#                         'Max_Dose_Gy': dvh.max,
#                         'Min_Dose_Gy': dvh.min,
#                         # Constraints: Volume receiving >= X Gy
#                         'V5Gy_%': dvh.volume_constraint(5).volume,
#                         'V10Gy_%': dvh.volume_constraint(10).volume,
#                         'V20Gy_%': dvh.volume_constraint(20).volume,
#                         'V30Gy_%': dvh.volume_constraint(30).volume,
#                         # Constraints: Dose received by X% of volume
#                         'D95_Gy': dvh.dose_constraint(95).value,
#                         'D50_Gy': dvh.dose_constraint(50).value
#                     }
#                     results_list.append(metrics)
#                 else:
#                     print(f"[{i+1}/{len(patient_folders)}] {patient_id}: [SKIP] No 'Lung' structure found.")
#                     skipped_patients.append(patient_id)
                    
#             except Exception as e:
#                 print(f"[{i+1}/{len(patient_folders)}] {patient_id}: [ERROR] {str(e)}")
#                 skipped_patients.append(patient_id)
#         else:
#             print(f"[{i+1}/{len(patient_folders)}] {patient_id}: [SKIP] Missing RTSTRUCT or RTDOSE.")
#             skipped_patients.append(patient_id)

# print("\n--- Processing Complete ---")

--- Starting Batch Extraction ---
[ERROR] Data path not found: ../Data/Local_Data/Processed/

--- Processing Complete ---


In [ ]:
# if results_list:
#     # Convert to DataFrame
#     df_results = pd.DataFrame(results_list)
    
#     # Save to CSV
#     df_results.to_csv(output_csv, index=False)
    
#     print(f"Success! Extracted features for {len(df_results)} patients.")
#     print(f"Results saved to: {output_csv}")
    
#     print("\n--- First 5 Rows ---")
#     display(df_results.head())
# else:
#     print("No features extracted. Check your data folders.")

# if skipped_patients:
#     print(f"\nSkipped {len(skipped_patients)} patients: {skipped_patients}")

No features extracted. Check your data folders.


In [ ]:
# import os
# import pandas as pd
# import pydicom
# from dicompylercore import dicomparser, dvhcalc
# import warnings

# # Suppress minor DICOM warnings to keep output clean
# warnings.filterwarnings("ignore")

# # --- 1. CONFIGURATION ---
# # IMPORTANT: Double check these paths match your folder structure exactly
# data_sources = {
#     'Ahsania': '../Data/AMCGH/',      
#     'Square': '../Data/SQUARE/'
# }

# output_csv = '../Results/local_dosiomics_multicenter.csv'

# # Create Results directory if it doesn't exist
# os.makedirs('../Results', exist_ok=True)

# print(f"--- STARTING EXTRACTION ---")
# print(f"Targets: {list(data_sources.keys())}")
# print(f"Output: {output_csv}")

# # --- 2. ROBUST SEARCH FUNCTION ---
# def find_lung_roi_robust(structure_dict):
#     """
#     Finds the Lung structure ID by checking common names.
#     Returns: (roi_id, roi_name) or (None, None)
#     """
#     # List of names to check (lowercase, no spaces)
#     target_names_priority = [
#         'lungs_combined', 'lungs-combined', 'lungs_total', 'lung_total', 
#         'total_lung', 'total lung', 'lungs', 'lung', 'lungs(total)', 'whole_lung',
#         'lung_l+r', 'lungs_l+r', 'both lungs', 'lung combined', 'lungs combined'
#     ]
    
#     # Map structure ID to a "clean" name
#     available_rois = {}
#     for key, struct in structure_dict.items():
#         # Clean: lowercase, remove spaces, underscores, dashes
#         clean = struct['name'].lower().strip().replace(' ', '').replace('_', '').replace('-', '')
#         available_rois[key] = clean

#     # Check for matches
#     for target in target_names_priority:
#         clean_target = target.replace(' ', '').replace('_', '').replace('-', '')
#         for key, clean_name in available_rois.items():
#             if clean_target == clean_name:
#                 return key, structure_dict[key]['name'] # Found it!
    
#     return None, None

# # --- 3. MAIN PROCESSING LOOP ---
# results_list = []

# for source_name, source_path in data_sources.items():
#     # Verify folder exists
#     if not os.path.exists(source_path):
#         print(f"\n[CRITICAL ERROR] Folder not found: {source_path}")
#         print(f"Current Working Directory: {os.getcwd()}")
#         continue
        
#     print(f"\n>>> Processing Hospital: {source_name} ...")
    
#     # Get patient folders
#     try:
#         patient_folders = sorted([f for f in os.listdir(source_path) if os.path.isdir(os.path.join(source_path, f))])
#     except Exception as e:
#         print(f"Error reading directory: {e}")
#         continue
        
#     print(f"Found {len(patient_folders)} patients.")
    
#     for i, patient_id in enumerate(patient_folders):
#         patient_dir = os.path.join(source_path, patient_id)
        
#         # A. Find Files (RTSTRUCT & RTDOSE)
#         rtstruct_path = None
#         rtdose_path = None
        
#         # Deep search using os.walk
#         for root, dirs, files in os.walk(patient_dir):
#             for f in files:
#                 full_path = os.path.join(root, f)
#                 try:
#                     # Quick header check
#                     dcm = pydicom.dcmread(full_path, stop_before_pixels=True)
#                     if dcm.Modality == 'RTSTRUCT':
#                         rtstruct_path = full_path
#                     elif dcm.Modality == 'RTDOSE':
#                         # Prefer 'Sum' or 'Total' dose files if available
#                         if rtdose_path is None: 
#                             rtdose_path = full_path
#                         elif 'sum' in f.lower() or 'total' in f.lower():
#                             rtdose_path = full_path
#                 except:
#                     continue # Skip non-dicom files
            
#             if rtstruct_path and rtdose_path:
#                 break # Found both, stop searching
        
#         # B. Extract Features
#         if rtstruct_path and rtdose_path:
#             try:
#                 # Load Structures
#                 rtss = dicomparser.DicomParser(rtstruct_path)
#                 structures = rtss.GetStructures()
                
#                 # Find Lungs
#                 roi_id, roi_name = find_lung_roi_robust(structures)
                
#                 if roi_id:
#                     # Calculate DVH (Heavy computation)
#                     dvh = dvhcalc.get_dvh(rtstruct_path, rtdose_path, roi_id)
                    
#                     if dvh:
#                         metrics = {
#                             'PatientID': patient_id,
#                             'Source': source_name,
#                             'Structure_Name': roi_name,
#                             'Mean_Dose_Gy': dvh.mean,
#                             'Max_Dose_Gy': dvh.max,
#                             'Min_Dose_Gy': dvh.min,
#                             'V20Gy_%': dvh.volume_constraint(20).volume,
#                             'D95_Gy': dvh.dose_constraint(95).value
#                         }
#                         results_list.append(metrics)
#                         print(f"  [{i+1}] {patient_id}: Success")
#                     else:
#                         print(f"  [{i+1}] {patient_id}: [FAIL] Empty DVH")
#                 else:
#                     # Log available structures for debugging
#                     avail = [s['name'] for k,s in structures.items()]
#                     print(f"  [{i+1}] {patient_id}: [SKIP] No Lung found. Available: {avail[:3]}")
                    
#             except Exception as e:
#                 print(f"  [{i+1}] {patient_id}: [ERROR] {str(e)}")
#         else:
#             print(f"  [{i+1}] {patient_id}: [SKIP] Missing RTSTRUCT or RTDOSE")

# # --- 4. SAVE ---
# if results_list:
#     df_results = pd.DataFrame(results_list)
#     df_results.to_csv(output_csv, index=False)
#     print(f"\n--- DONE ---")
#     print(f"Saved {len(df_results)} patients to: {output_csv}")
#     print(df_results.head())
# else:
#     print("\n[WARNING] No data extracted.")

--- STARTING EXTRACTION ---
Targets: ['Ahsania', 'Square']
Output: ../Results/local_dosiomics_multicenter.csv

>>> Processing Hospital: Ahsania ...
Found 35 patients.
  [1] 1042new: [SKIP] Missing RTSTRUCT or RTDOSE
  [2] 1083New: [SKIP] Missing RTSTRUCT or RTDOSE
  [3] 1102 new: [SKIP] Missing RTSTRUCT or RTDOSE
  [4] 1112med: [SKIP] Missing RTSTRUCT or RTDOSE
  [5] 1115new: [SKIP] Missing RTSTRUCT or RTDOSE
  [6] 1127new: [SKIP] Missing RTSTRUCT or RTDOSE
  [7] 1161new: [SKIP] Missing RTSTRUCT or RTDOSE
  [8] 1209new: [SKIP] Missing RTSTRUCT or RTDOSE
  [9] 1233new: [SKIP] Missing RTSTRUCT or RTDOSE
  [10] 1299new: [SKIP] Missing RTSTRUCT or RTDOSE
  [11] 1342new: [SKIP] Missing RTSTRUCT or RTDOSE
  [12] 1356new: [SKIP] Missing RTSTRUCT or RTDOSE
  [13] 1392new: [SKIP] Missing RTSTRUCT or RTDOSE
  [14] 1430new: [SKIP] Missing RTSTRUCT or RTDOSE
  [15] 1495new: [SKIP] Missing RTSTRUCT or RTDOSE
  [16] 20230367new: [SKIP] Missing RTSTRUCT or RTDOSE
  [17] 20230396(39): [SKIP] Missing R

In [ ]:
# import os
# import pandas as pd
# import pydicom
# from dicompylercore import dicomparser, dvhcalc
# import warnings

# # Suppress warnings
# warnings.filterwarnings("ignore")

# # Define AMCGH
# data_sources = {
#     'Ahsania': '../Data/AMCGH/' 
# }

# output_csv = '../Results/amcgh_dosiomics.csv'
# os.makedirs('../Results', exist_ok=True)

# print("Configured for AMCGH extraction.")

Configured for AMCGH extraction.


In [ ]:
# def find_lung_roi_robust(structure_dict):
#     """
#     Finds Lung structure ID by checking common names.
#     """
#     target_names_priority = [
#         'lungs_combined', 'lungs-combined', 'lungs_total', 'lung_total', 
#         'total_lung', 'total lung', 'lungs', 'lung', 'lungs(total)', 'whole_lung',
#         'lung_l+r', 'lungs_l+r', 'both lungs', 'lung combined'
#     ]
    
#     available_rois = {}
#     for key, struct in structure_dict.items():
#         clean = struct['name'].lower().strip().replace(' ', '').replace('_', '').replace('-', '')
#         available_rois[key] = clean

#     for target in target_names_priority:
#         clean_target = target.replace(' ', '').replace('_', '').replace('-', '')
#         for key, clean_name in available_rois.items():
#             if clean_target == clean_name:
#                 return key, structure_dict[key]['name']
    
#     return None, None

In [ ]:
# results_list = []

# for source_name, source_path in data_sources.items():
#     if not os.path.exists(source_path):
#         print(f"[ERROR] Folder not found: {source_path}")
#         continue
        
#     patient_folders = sorted([f for f in os.listdir(source_path) if os.path.isdir(os.path.join(source_path, f))])
#     print(f"Processing {len(patient_folders)} patients in {source_name}...")
    
#     for i, patient_id in enumerate(patient_folders):
#         patient_dir = os.path.join(source_path, patient_id)
        
#         rtstruct_path = None
#         rtdose_path = None
        
#         # Deep search
#         for root, dirs, files in os.walk(patient_dir):
#             for f in files:
#                 full_path = os.path.join(root, f)
#                 try:
#                     # CRITICAL FIX: force=True allows reading files without standard headers
#                     dcm = pydicom.dcmread(full_path, stop_before_pixels=True, force=True)
                    
#                     # Safe check for modality
#                     mod = dcm.get("Modality", "Unknown")
                    
#                     if mod == 'RTSTRUCT':
#                         rtstruct_path = full_path
#                     elif mod == 'RTDOSE':
#                         # Prefer 'Sum' or 'Total' dose files
#                         if rtdose_path is None: 
#                             rtdose_path = full_path
#                         elif 'sum' in f.lower() or 'total' in f.lower():
#                             rtdose_path = full_path
#                 except:
#                     continue
            
#             if rtstruct_path and rtdose_path:
#                 break 
        
#         # Extract features
#         if rtstruct_path and rtdose_path:
#             try:
#                 # Force read here as well for dicompyler compatibility
#                 # Note: dicompylercore handles files well, but we pass paths
#                 rtss = dicomparser.DicomParser(rtstruct_path)
#                 structures = rtss.GetStructures()
#                 roi_id, roi_name = find_lung_roi_robust(structures)
                
#                 if roi_id:
#                     dvh = dvhcalc.get_dvh(rtstruct_path, rtdose_path, roi_id)
#                     if dvh:
#                         metrics = {
#                             'PatientID': patient_id,
#                             'Source': source_name,
#                             'Structure_Name': roi_name,
#                             'Mean_Dose_Gy': dvh.mean,
#                             'V20Gy_%': dvh.volume_constraint(20).volume
#                         }
#                         results_list.append(metrics)
#                         print(f"[{i+1}] {patient_id}: Success ({roi_name})")
#                     else:
#                         print(f"[{i+1}] {patient_id}: [FAIL] Empty DVH")
#                 else:
#                     avail = [s['name'] for k,s in structures.items()]
#                     print(f"[{i+1}] {patient_id}: [SKIP] No Lung found. Avail: {avail[:3]}")
#             except Exception as e:
#                 print(f"[{i+1}] {patient_id}: [ERROR] {str(e)}")
#         else:
#             print(f"[{i+1}] {patient_id}: [SKIP] Missing Files (Struct: {rtstruct_path is not None}, Dose: {rtdose_path is not None})")

Processing 52 patients in Ahsania...
[1] 1042new: [SKIP] No Lung found. Avail: ['patient', 'GTV_3600/18', 'CTV_3600/18']
[2] 1083New: [SKIP] No Lung found. Avail: ['patient', 'gtv t', 'heart']
[3] 1102 new: [SKIP] No Lung found. Avail: ['patient', 'GTV_6000/30', 'Rt Lung']
[4] 1115new: [SKIP] No Lung found. Avail: ['patient', 'ptv 6000/30', 'spc']
[5] 1127new: [SKIP] No Lung found. Avail: ['patient', 'gtv t', 'heart']
[6] 1161new: [SKIP] No Lung found. Avail: ['patient', 'Lung V20', 'Heart']
[7] 1209new: [SKIP] No Lung found. Avail: ['patient', 'gtv_6000/30', 'rt lung']
[8] 1233new: [SKIP] No Lung found. Avail: ['patient', 'gtv t', 'ctv t']
[9] 1299new: [SKIP] No Lung found. Avail: ['patient', 'gtv t', 'ctv t']
[10] 1342new: [SKIP] No Lung found. Avail: ['patient', 'GTV', 'CTV']
[11] 1356new: [SKIP] No Lung found. Avail: ['patient', 'gtv', 'gtv t']
[12] 1392new: [SKIP] No Lung found. Avail: ['patient', 'gtv', 'ctv']
[13] 1430new: [SKIP] No Lung found. Avail: ['patient', 'GTV T', 'GTV N

In [ ]:
# import os
# import pydicom

# # --- CONFIGURATION ---
# # We point specifically to the first patient's folder
# # Make sure this path matches exactly where your data is
# base_path = '../Data/AMCGH/'
# test_patient_id = '1042new'  # Based on your output
# patient_dir = os.path.join(base_path, test_patient_id)

# print(f"--- FORENSIC INSPECTION OF: {patient_dir} ---")

# if not os.path.exists(patient_dir):
#     print("CRITICAL: The patient folder path itself does not exist.")
# else:
#     file_count = 0
#     # Walk through every file
#     for root, dirs, files in os.walk(patient_dir):
#         print(f"\nScanning Subfolder: {root}")
#         for f in files:
#             file_count += 1
#             full_path = os.path.join(root, f)
            
#             print(f"  File: {f}")
            
#             # Try to read DICOM header
#             try:
#                 # force=True allows reading files without .dcm extension
#                 dcm = pydicom.dcmread(full_path, stop_before_pixels=True, force=True)
                
#                 # Extract key tags
#                 modality = dcm.get("Modality", "Unknown")
#                 study_desc = dcm.get("StudyDescription", "No Desc")
#                 series_desc = dcm.get("SeriesDescription", "No Desc")
                
#                 print(f"    -> DICOM VALID | Modality: {modality} | Series: {series_desc}")
                
#             except Exception as e:
#                 print(f"    -> NOT DICOM or Read Error: {str(e)}")
            
#             # Limit to first 20 files to avoid flooding the screen
#             if file_count >= 20:
#                 print("\n... Stopping inspection after 20 files ...")
#                 break
#         if file_count >= 20: break

#     if file_count == 0:
#         print("\n[RESULT] The folder is empty!")

--- FORENSIC INSPECTION OF: ../Data/AMCGH/1042new ---

Scanning Subfolder: ../Data/AMCGH/1042new
  File: 20251042_CT3_image00000.DCM
    -> DICOM VALID | Modality: CT | Series:  Body 5.0  CE
  File: 20251042_CT3_image00001.DCM
    -> DICOM VALID | Modality: CT | Series:  Body 5.0  CE
  File: 20251042_CT3_image00002.DCM
    -> DICOM VALID | Modality: CT | Series:  Body 5.0  CE
  File: 20251042_CT3_image00003.DCM
    -> DICOM VALID | Modality: CT | Series:  Body 5.0  CE
  File: 20251042_CT3_image00004.DCM
    -> DICOM VALID | Modality: CT | Series:  Body 5.0  CE
  File: 20251042_CT3_image00005.DCM
    -> DICOM VALID | Modality: CT | Series:  Body 5.0  CE
  File: 20251042_CT3_image00006.DCM
    -> DICOM VALID | Modality: CT | Series:  Body 5.0  CE
  File: 20251042_CT3_image00007.DCM
    -> DICOM VALID | Modality: CT | Series:  Body 5.0  CE
  File: 20251042_CT3_image00008.DCM
    -> DICOM VALID | Modality: CT | Series:  Body 5.0  CE
  File: 20251042_CT3_image00009.DCM
    -> DICOM VALID | 

In [ ]:
# import os
# import pydicom

# # --- CONFIGURATION ---
# base_path = '../Data/AMCGH/'
# test_patient_id = '1042new'
# patient_dir = os.path.join(base_path, test_patient_id)

# print(f"--- FULL CENSUS OF: {patient_dir} ---")

# modality_counts = {}
# non_ct_files = []

# for root, dirs, files in os.walk(patient_dir):
#     for f in files:
#         full_path = os.path.join(root, f)
#         try:
#             # force=True ensures we read even non-standard headers
#             dcm = pydicom.dcmread(full_path, stop_before_pixels=True, force=True)
            
#             # Get Modality
#             mod = dcm.get("Modality", "Unknown")
            
#             # Count it
#             modality_counts[mod] = modality_counts.get(mod, 0) + 1
            
#             # If it's NOT a CT image, save the filename to show the user
#             if mod != 'CT':
#                 non_ct_files.append((f, mod))
                
#         except:
#             # Ignore non-dicom files
#             pass

# print("\n--- RESULTS ---")
# print(f"Total Files Scanned: {sum(modality_counts.values())}")
# print("\nModality Breakdown:")
# for mod, count in modality_counts.items():
#     print(f"  {mod}: {count} files")

# print("\n--- NON-CT FILES FOUND ---")
# if non_ct_files:
#     for name, mod in non_ct_files:
#         print(f"  File: {name} -> {mod}")
# else:
#     print("  [CRITICAL] No RTSTRUCT or RTDOSE files found! Only CTs.")

--- FULL CENSUS OF: ../Data/AMCGH/1042new ---

--- RESULTS ---
Total Files Scanned: 136

Modality Breakdown:
  CT: 133 files
  RTPLAN: 1 files
  RTDOSE: 1 files
  RTSTRUCT: 1 files

--- NON-CT FILES FOUND ---
  File: 20251042_Plan36.dcm -> RTPLAN
  File: 20251042_Plan36_Dose.dcm -> RTDOSE
  File: 20251042_StrctrSets.dcm -> RTSTRUCT


In [1]:
import os
import pandas as pd
import pydicom
from dicompylercore import dicomparser, dvhcalc
import warnings

# Suppress warnings
warnings.filterwarnings("ignore")

# --- 1. CONFIGURATION ---
data_sources = {
    'Ahsania': '../Data/AMCGH/' 
}

output_csv = '../Results/amcgh_dosiomics.csv'
os.makedirs('../Results', exist_ok=True)

print(f"--- STARTING AHSANIA TARGET EXTRACTION ---")

# --- 2. TARGET SEARCH FUNCTION ---
def find_target_roi_ahsania(structure_dict):
    """
    Prioritizes Tumor Targets (GTV > CTV > PTV).
    Returns: (roi_id, roi_name, roi_type)
    """
    # Priority 1: GTV (Gross Tumor Volume) - Most common in your logs
    gtv_names = ['gtv', 'gtv_t', 'gtv t', 'gtv_total', 'gtv total']
    
    # Priority 2: CTV (Clinical Target Volume)
    ctv_names = ['ctv', 'ctv_t', 'ctv t', 'ctv_total']

    # Priority 3: PTV (Planning Target Volume)
    ptv_names = ['ptv', 'ptv_total', 'ptv total', 'ptv_6000']

    # Clean available names for comparison
    available_rois = {}
    for key, struct in structure_dict.items():
        clean = struct['name'].lower().strip().replace(' ', '').replace('_', '').replace('-', '')
        available_rois[key] = clean

    # Check GTV
    for name in gtv_names:
        clean_target = name.replace(' ', '').replace('_', '').replace('-', '')
        for key, clean_avail in available_rois.items():
            if clean_target in clean_avail: return key, structure_dict[key]['name'], 'Target_GTV'

    # Check CTV
    for name in ctv_names:
        clean_target = name.replace(' ', '').replace('_', '').replace('-', '')
        for key, clean_avail in available_rois.items():
            if clean_target in clean_avail: return key, structure_dict[key]['name'], 'Target_CTV'

    # Check PTV
    for name in ptv_names:
        clean_target = name.replace(' ', '').replace('_', '').replace('-', '')
        for key, clean_avail in available_rois.items():
            if clean_target in clean_avail: return key, structure_dict[key]['name'], 'Target_PTV'

    return None, None, None

# --- 3. MAIN LOOP ---
results_list = []

for source_name, source_path in data_sources.items():
    if not os.path.exists(source_path):
        print(f"[ERROR] Folder not found: {source_path}")
        continue
        
    patient_folders = sorted([f for f in os.listdir(source_path) if os.path.isdir(os.path.join(source_path, f))])
    print(f"Processing {len(patient_folders)} patients in {source_name}...")
    
    for i, patient_id in enumerate(patient_folders):
        patient_dir = os.path.join(source_path, patient_id)
        
        rtstruct_path = None
        rtdose_path = None
        
        # Deep search with FORCE=TRUE
        for root, dirs, files in os.walk(patient_dir):
            for f in files:
                full_path = os.path.join(root, f)
                try:
                    dcm = pydicom.dcmread(full_path, stop_before_pixels=True, force=True)
                    mod = dcm.get("Modality", "Unknown")
                    if mod == 'RTSTRUCT':
                        rtstruct_path = full_path
                    elif mod == 'RTDOSE':
                        if rtdose_path is None: rtdose_path = full_path
                        elif 'sum' in f.lower() or 'total' in f.lower(): rtdose_path = full_path
                except:
                    continue
            if rtstruct_path and rtdose_path: break 
        
        # Extract
        if rtstruct_path and rtdose_path:
            try:
                rtss = dicomparser.DicomParser(rtstruct_path)
                structures = rtss.GetStructures()
                
                # Find GTV/CTV/PTV
                roi_id, roi_name, roi_type = find_target_roi_ahsania(structures)
                
                if roi_id:
                    dvh = dvhcalc.get_dvh(rtstruct_path, rtdose_path, roi_id)
                    if dvh:
                        metrics = {
                            'PatientID': patient_id,
                            'Source': source_name,
                            'ROI_Name': roi_name,
                            'ROI_Type': roi_type,
                            'Mean_Dose_Gy': dvh.mean,
                            'Max_Dose_Gy': dvh.max,
                            'Min_Dose_Gy': dvh.min,
                            'D95_Gy': dvh.dose_constraint(95).value # Critical for Target Coverage
                        }
                        results_list.append(metrics)
                        print(f"[{i+1}] {patient_id}: Success ({roi_name} -> {roi_type})")
                    else:
                        print(f"[{i+1}] {patient_id}: [FAIL] Empty DVH")
                else:
                    avail = [s['name'] for k,s in structures.items()]
                    print(f"[{i+1}] {patient_id}: [SKIP] No Target Found. Avail: {avail[:3]}")
            except Exception as e:
                print(f"[{i+1}] {patient_id}: [ERROR] {str(e)}")
        else:
            print(f"[{i+1}] {patient_id}: [SKIP] Missing Files")

# --- 4. SAVE ---
if results_list:
    df = pd.DataFrame(results_list)
    df.to_csv(output_csv, index=False)
    print(f"\n--- SUCCESS ---")
    print(f"Saved {len(df)} patients to {output_csv}")
    print(df.head())
else:
    print("\nNo data extracted.")

--- STARTING AHSANIA TARGET EXTRACTION ---
Processing 52 patients in Ahsania...
[1] 1042new: Success (GTV_3600/18 -> Target_GTV)
[2] 1083New: Success (gtv t -> Target_GTV)
[3] 1102 new: Success (GTV_6000/30 -> Target_GTV)
[4] 1115new: Success (ptv 6000/30 -> Target_PTV)
[5] 1127new: Success (gtv t -> Target_GTV)
[6] 1161new: Success (CTV T -> Target_CTV)
[7] 1209new: Success (gtv_6000/30 -> Target_GTV)
[8] 1233new: Success (gtv t -> Target_GTV)
[9] 1299new: Success (gtv t -> Target_GTV)
[10] 1342new: Success (GTV -> Target_GTV)
[11] 1356new: Success (gtv -> Target_GTV)
[12] 1392new: Success (gtv -> Target_GTV)
[13] 1430new: Success (GTV T -> Target_GTV)
[14] 1495new: Success (ptv_6000/30 -> Target_PTV)
[15] 20230367new: Success (GTV -> Target_GTV)
[16] 20230396(39): Success (gtv_3900/13 -> Target_GTV)
[17] 20230490new: Success (GTV -> Target_GTV)
[18] 20250013: Success (gtv t -> Target_GTV)
[19] 20250106: Success (GTV T -> Target_GTV)
[20] 20250117: Success (gtv t -> Target_GTV)
[21] 2